In [1]:
from pathlib import Path
import os,json,ast,hashlib,time,random
os.environ['CUBLAS_WORKSPACE_CONFIG']=':4096:8'
import numpy as np,h5py,torch
from torch import nn
from torch.utils.data import DataLoader,TensorDataset
from sklearn.metrics import roc_auc_score,average_precision_score
ROOT=Path(r'C:/Users/choud/Desktop/Minor project');OUT=ROOT/'model_improvement/attention_tuning_v1';V3=ROOT/'cnn/cnn wave/eew_causal_v3';CR=V3/'cnn_eew_training';BR=ROOT/'bilstm_eew/bilstm_training_v1';TAB=ROOT/'bilstm_eew/causal_tabular_v1/tabular_features.h5'
DEV=torch.device('cuda' if torch.cuda.is_available() else 'cpu');torch.set_num_threads(4);torch.use_deterministic_algorithms(True);torch.backends.cudnn.benchmark=False
CFG={'seed':42,'epochs':35,'patience':7,'min_delta':1e-4,'batch_size':256,'lr':.001,'weight_decay':.0001,'dropout':.2,'clip':5.,'low_fpr_target':.05}
def sha(p):
 h=hashlib.sha256()
 with p.open('rb') as f:
  for b in iter(lambda:f.read(4194304),b''):h.update(b)
 return h.hexdigest()
# Import exact class definitions only: no original setup, training or evaluation executes.
for path,cell,name in [(CR/'01_cnn_training.ipynb',1,'EarlyCNN'),(BR/'01_bilstm_training.ipynb',0,'TabularBiLSTM')]:
 tree=ast.parse(''.join(json.loads(path.read_text(encoding='utf-8'))['cells'][cell]['source']));node=next(x for x in tree.body if isinstance(x,ast.ClassDef) and x.name==name);exec(compile(ast.Module(body=[node],type_ignores=[]),'original_class_only','exec'))
class FusionHead(nn.Module):
 def __init__(self):
  super().__init__();self.shared=nn.Sequential(nn.Linear(64,32),nn.SiLU(),nn.Dropout(.2));self.detection=nn.Linear(32,1);self.magnitude=nn.Linear(32,1)
 def forward(self,z):
  z=self.shared(z);return self.detection(z)[:,0],self.magnitude(z)[:,0]
def align():
 mapping={};parents={};summary={}
 with h5py.File(TAB,'r') as tab:
  for split,n in [('train',11324),('validation',2426)]:
   with h5py.File(V3/'dataset'/f'eew_dataset_{split}.h5','r') as w:
    t=tab[split];wp=w['parent_id'].asstr()[:];tp=t['parent_id'].asstr()[:];assert len(wp)==len(tp)==n and len(set(wp))==len(set(tp))==n,'Critical parent uniqueness/count failure'
    lookup={p:i for i,p in enumerate(tp)};assert set(wp)==set(tp),'Parent set mismatch';ix=np.array([lookup[p] for p in wp]);mapping[split]=ix;parents[split]=set(wp)
    for key in ['source_id','parent_id','pair_id','combined_index','labels','magnitude','magnitude_mask']+[f'{k}_{h}s' for h in [1,2,3] for k in ['pre_s_valid','classification_mask','magnitude_mask']]:
     a=w[key].asstr()[:] if w[key].dtype.kind=='O' else w[key][:];b=t[key].asstr()[:] if t[key].dtype.kind=='O' else t[key][:];assert np.array_equal(a,b[ix]),f'Alignment failure: {split}/{key}'
    assert w.attrs['sampling_rate_hz']==tab.attrs['sampling_rate_hz']==100 and w.attrs['pre_context_samples']==tab.attrs['pre_reference_samples']==100
    for h in [1,2,3]:
     assert w[f'windows_{h}s_norm'].shape==(n,100+h*100) and t[f'{h}s/X'].shape==(n,10+10*h,7)
     assert t[f'{h}s'].attrs['last_bin_end_relative_sample_exclusive']==h*100
     y=w['labels'][:];r=w[f'magnitude_mask_{h}s'][:];k=w[f'classification_mask_{h}s'][:];assert np.array_equal(r,(y==1)&k) and np.array_equal(r,w[f'pre_s_valid_{h}s'][:]) and np.all(w['magnitude'][:][y==0]==-1)
    summary[split]={'records':n,'reordered_tabular_rows':int((ix!=np.arange(n)).sum()),'parent_order_sha256':hashlib.sha256('\n'.join(wp).encode()).hexdigest()}
  for a,b in [('train','validation')]:assert not(parents[a]&parents[b]),'Cross-split parent overlap'
 return mapping,summary
MAP,ALIGNMENT=align()
def branches(h):
 cc=torch.load(CR/'models'/f'cnn_{h}s_best.pt',map_location='cpu',weights_only=False);bc=torch.load(BR/'models'/f'bilstm_{h}s_best.pt',map_location='cpu',weights_only=False)
 assert cc['horizon_seconds']==bc['horizon']==h
 c=EarlyCNN().to(DEV);b=TabularBiLSTM().to(DEV);c.load_state_dict(cc['state_dict'],strict=True);b.load_state_dict(bc['state_dict'],strict=True)
 for model in [c,b]:model.requires_grad_(False);model.eval()
 return c,b,cc,bc
@torch.inference_mode()
def encode(split,h,c,b,cc,bc,only_common=False):
 with h5py.File(V3/'dataset'/f'eew_dataset_{split}.h5','r') as w,h5py.File(TAB,'r') as t:
  keep=w[f'classification_mask_{h}s'][:].astype(bool);common=np.logical_and.reduce([w[f'classification_mask_{j}s'][:] for j in [1,2,3]])
  assert np.array_equal(common,t[split]['common_cohort_mask'][:][MAP[split]])
  if only_common:keep &= common
  x=w[f'windows_{h}s_norm'][:][keep,None,:];tab=t[split][f'{h}s/X'][:][MAP[split]][keep]
  y=w['labels'][:][keep].astype('float32');m=w['magnitude'][:][keep];reg=w[f'magnitude_mask_{h}s'][:][keep];ids=w['source_id'].asstr()[:][keep]
  assert np.isfinite(x).all() and np.isfinite(tab).all()
 embeddings=[];cp=[];cm=[];bp=[];bm=[];captured={}
 hc=c.shared.register_forward_hook(lambda module,args,out:captured.update(c=out));hb=b.shared.register_forward_hook(lambda module,args,out:captured.update(b=out))
 try:
  for start in range(0,len(y),256):
   ca,cv=c(torch.from_numpy(x[start:start+256]).to(DEV));ba,bv=b(torch.from_numpy(tab[start:start+256]).to(DEV));embeddings.append(torch.cat([captured['c'],captured['b']],1).cpu());cp.append(ca.sigmoid().cpu());cm.append(cv.cpu());bp.append(ba.sigmoid().cpu());bm.append(bv.cpu())
 finally:hc.remove();hb.remove()
 z=torch.cat(embeddings);assert z.shape==(len(y),64) and not z.requires_grad
 return dict(z=z,y=torch.from_numpy(y),m=torch.from_numpy(m),reg=torch.from_numpy(reg),ids=ids,cnn_p=torch.cat(cp).numpy(),cnn_m=torch.cat(cm).numpy()*cc['target_scale']+cc['target_center'],bilstm_p=torch.cat(bp).numpy(),bilstm_m=torch.cat(bm).numpy()*bc['scale']+bc['center'])
def det(y,p,t=.5):
 y=np.asarray(y)==1;q=p>=t;tp=int((y&q).sum());fp=int((~y&q).sum());tn=int((~y&~q).sum());fn=int((y&~q).sum())
 return dict(accuracy=(tp+tn)/len(y),precision=tp/(tp+fp) if tp+fp else 0.,recall=tp/(tp+fn),f1=2*tp/(2*tp+fp+fn),false_positive_rate=fp/(fp+tn),roc_auc=float(roc_auc_score(y,p)),pr_auc_ap=float(average_precision_score(y,p)),confusion_matrix=[[tn,fp],[fn,tp]])
def magnitude(a,p):
 a=a.astype(float);p=p.astype(float);e=p-a;strong=a>=5
 return dict(n=len(a),mae=float(np.abs(e).mean()),rmse=float(np.sqrt(np.mean(e**2))),r2=float(1-np.sum(e**2)/np.sum((a-a.mean())**2)),bias=float(e.mean()),actual_variance=float(a.var()),predicted_variance=float(p.var()),std_ratio=float(p.std()/a.std()),slope=float(np.polyfit(a,p,1)[0]),strong_n=int(strong.sum()),strong_underprediction_percent=float((e[strong]<0).mean()*100),strong_bias=float(e[strong].mean()))

class AttentionHead(nn.Module):
 def __init__(self):
  super().__init__();self.attention=nn.Sequential(nn.Linear(64,32),nn.Tanh(),nn.Linear(32,1,bias=False));self.shared=nn.Sequential(nn.Linear(96,32),nn.SiLU(),nn.Dropout(.2));self.detection=nn.Linear(32,1);self.magnitude=nn.Linear(32,1)
 def forward(self,c,s):
  weights=self.attention(s).squeeze(-1).softmax(dim=1);z=self.shared(torch.cat([c,(s*weights.unsqueeze(-1)).sum(1)],1));return self.detection(z)[:,0],self.magnitude(z)[:,0],weights
@torch.inference_mode()
def temporal(split,h,c,b,only_common=False):
 with h5py.File(V3/'dataset'/f'eew_dataset_{split}.h5','r') as w,h5py.File(TAB,'r') as t:
  keep=w[f'classification_mask_{h}s'][:].astype(bool)
  if only_common:keep &= np.logical_and.reduce([w[f'classification_mask_{j}s'][:] for j in [1,2,3]])
  x=w[f'windows_{h}s_norm'][:][keep,None,:];v=t[split][f'{h}s/X'][:][MAP[split]][keep]
  d={k:torch.from_numpy(w[key][:][keep]) for k,key in [('y','labels'),('m','magnitude'),('reg',f'magnitude_mask_{h}s')]};d['y']=d['y'].float();d['reg']=d['reg'].bool();d['ids']=w['source_id'].asstr()[:][keep]
 cs=[];ss=[]
 for j in range(0,len(x),256):
  xx=torch.from_numpy(x[j:j+256]).to(DEV);q=c.features(torch.sign(xx)*torch.log1p(xx.abs()));cs.append(c.shared(torch.cat([q.mean(-1),q.amax(-1)],1)).cpu());s,_=b.lstm(torch.from_numpy(v[j:j+256]).to(DEV));ss.append(s.cpu())
 d['c']=torch.cat(cs);d['s']=torch.cat(ss);assert d['s'].shape==(len(x),10+10*h,64);assert torch.isfinite(d['s']).all();return d

def loader(d,ck,shuffle=False):
 return DataLoader(TensorDataset((d['c']-ck['cmean'])/ck['cscale'],(d['s']-ck['smean'])/ck['sscale'],d['y'],d['m'],d['reg']),batch_size=256,shuffle=shuffle,generator=torch.Generator().manual_seed(42))
def epoch(head,data,ck,opt=None):
 head.train(opt is not None);ds=ms=0.;n=nr=0
 with torch.set_grad_enabled(opt is not None):
  for c,s,y,m,r in data:
   c,s,y,m,r=[v.to(DEV) for v in [c,s,y,m,r]]
   if opt is not None:opt.zero_grad(set_to_none=True)
   a,b,w=head(c,s);dl=nn.functional.binary_cross_entropy_with_logits(a,y,reduction='sum');ml=nn.functional.huber_loss(b[r],(m[r]-ck['target_center'])/ck['target_scale'],reduction='sum') if r.any() else b.sum()*0;loss=dl/len(y)+ml/r.sum().clamp_min(1);assert torch.isfinite(loss)
   if opt is not None:loss.backward();nn.utils.clip_grad_norm_(head.parameters(),5);opt.step()
   ds+=dl.item();ms+=ml.item();n+=len(y);nr+=int(r.sum())
 return dict(total=ds/n+ms/max(nr,1),detection=ds/n,magnitude=ms/max(nr,1))
@torch.inference_mode()
def predict_attention(head,d,ck):
 head.eval();ps=[];ms=[];ws=[]
 for c,s,y,m,r in loader(d,ck):
  p,m,w=head(c.to(DEV),s.to(DEV));ps.append(p.sigmoid().cpu());ms.append(m.cpu());ws.append(w.cpu())
 return torch.cat(ps).numpy(),torch.cat(ms).numpy()*ck['target_scale']+ck['target_center'],torch.cat(ws).numpy()
print('Exact paired inputs; frozen temporal backbone states available.')

import csv,copy
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
ORIGINAL=ROOT/'fusion/cnn_bilstm_attention_v1'
class TunedHead(nn.Module):
 def __init__(self,hidden=64,dropout=.2):
  super().__init__();self.attention=nn.Sequential(nn.Linear(64,32),nn.Tanh(),nn.Linear(32,1,bias=False));self.shared=nn.Sequential(nn.Linear(96,hidden),nn.SiLU(),nn.Dropout(dropout));self.detection=nn.Linear(hidden,1);self.magnitude=nn.Linear(hidden,1)
 def forward(self,c,s):
  w=self.attention(s).squeeze(-1).softmax(1);z=self.shared(torch.cat([c,(s*w.unsqueeze(-1)).sum(1)],1));return self.detection(z)[:,0],self.magnitude(z)[:,0],w
# Entire plan fixed before validation threshold analysis or trial training.
PLAN=[dict(trial=i+1,lr=lr,dropout=dp,hidden=hid,reg_weight=rw,variant='standard') for i,(lr,dp,hid,rw) in enumerate([
 (.0001,.1,64,1.),(.0003,.2,64,1.),(.001,.3,64,1.),(.0001,.2,128,1.),(.0003,.3,128,1.),(.001,.4,128,1.),(.0003,.1,64,.5),(.0003,.3,128,.5),(.0003,.2,64,2.),(.001,.2,128,2.)])]
PROTOCOL={'max_trials':12,'seed':42,'epochs':35,'patience':7,'min_delta':.0001,'threshold_grid':'0.01 through0.99 step0.01 plus0,1 and nextafter(max probability,+infinity)','operating_point':'Among FPR<=0.30 and recall>=0.70 maximize F1; ties lower FPR then closer0.5. If infeasible maximize recall under FPR<=0.30; label infeasible. Also unconstrained maximum-F1 threshold reported.','selection':'Gate feasible operating point, overall validation MAE <= original+0.02, strong MAE <=original+0.10. Useful improvement: FPR decreases >=0.02 with F1 >=original-0.01, OR F1 increases >=0.01 with FPR<=original+0.01, OR MAE decreases>=0.02 with FPR<=original+0.01 and F1>=original-0.01. Within useful candidates lexicographic lowest FPR, highest F1, lowest overall MAE, lowest strong MAE, lowest strong-underprediction percentage, trial ID. Otherwise retain original.','variants':'If selected phase-B configuration validation SD ratio<0.8, train exactly two from scratch with same configuration: strong_bin (Huber weight2 for training magnitude>=5), asymmetric (Huber weight2 when training magnitude>=5 and predicted<target). Phase B reference uses same feasible/error gates and lexicographic ranking, falling back to lowest overall validation MAE if none qualify. No validation examples in loss.','checkpoint_selection':'Unweighted validation BCE + standardized masked Huber for every trial; optimizer uses predefined regression weight/variant; identical fixed checkpoint criterion.'}
def stats(d,p,m):
 y=d['y'].numpy();a=d['m'].numpy();r=d['reg'].numpy();mm=magnitude(a[r],m[r]);strong=r&(a>=5);mm['strong_mae']=float(abs(m[strong]-a[strong]).mean());return y,mm
def operating(y,p):
 grid=np.unique(np.r_[0.,np.arange(1,100)/100,1.,float(np.nextafter(p.max(),np.float32(np.inf)))])
 table=[dict(threshold=float(t),**det(y,p,t)) for t in grid]
 best=max(table,key=lambda q:(q['f1'],-abs(q['threshold']-.5),-q['threshold']))
 feasible=[q for q in table if q['false_positive_rate']<=.30 and q['recall']>=.70]
 op=max(feasible,key=lambda q:(q['f1'],-q['false_positive_rate'],-abs(q['threshold']-.5))) if feasible else max([q for q in table if q['false_positive_rate']<=.30],key=lambda q:(q['recall'],q['f1'],-q['false_positive_rate']))
 return {'fixed':det(y,p,.5),'unconstrained':best,'operating':op,'feasible':bool(feasible),'curve':table}
def evaluate(head,d,ck):
 p,m,_=predict_attention(head,d,ck);y,mm=stats(d,p,m);return {**operating(y,p),'magnitude':mm}
def fit_epoch(head,data,ck,cfg,opt):
 head.train();total=0.;n=0
 for c,s,y,m,r in data:
  c,s,y,m,r=[v.to(DEV) for v in [c,s,y,m,r]];opt.zero_grad(set_to_none=True);logit,pred,_=head(c,s);cl=nn.functional.binary_cross_entropy_with_logits(logit,y)
  if r.any():
   target=(m[r]-ck['target_center'])/ck['target_scale'];raw=nn.functional.huber_loss(pred[r],target,reduction='none');weights=torch.ones_like(raw)
   if cfg['variant']=='strong_bin':weights=torch.where(m[r]>=5,2.,1.)
   if cfg['variant']=='asymmetric':weights=torch.where((m[r]>=5)&(pred[r]<target),2.,1.)
   ml=(raw*weights).sum()/weights.sum()
  else:ml=pred.sum()*0
  loss=cl+cfg['reg_weight']*ml;assert torch.isfinite(loss);loss.backward();nn.utils.clip_grad_norm_(head.parameters(),5.);opt.step();total+=float(loss.detach())*len(y);n+=len(y)
 return total/n


Exact paired inputs; frozen temporal backbone states available.


In [2]:

ck=torch.load(ORIGINAL/'models/attention_3s_best.pt',map_location='cpu',weights_only=False)
paths=[TAB,V3/'dataset/eew_dataset_train.h5',V3/'dataset/eew_dataset_validation.h5',V3/'dataset/eew_dataset_test.h5',CR/'models/cnn_3s_best.pt',BR/'models/bilstm_3s_best.pt',ORIGINAL/'models/attention_3s_best.pt',ORIGINAL/'01_attention_training.ipynb',ORIGINAL/'02_attention_evaluation.ipynb',ORIGINAL/'final_attention_report.md',ROOT/'final_comparison/four_model_eew_v1/final_comparison_report.md',CR/'01_cnn_training.ipynb',BR/'01_bilstm_training.ipynb']
before={str(p):sha(p) for p in paths};c,b,cc,bc=branches(3);tr=temporal('train',3,c,b);va=temporal('validation',3,c,b)
for key,prefix,dims in [('c','c',(0,)),('s','s',(0,1))]:
 mean=tr[key].mean(dims);sd=tr[key].std(dims,unbiased=False);sd=torch.where(sd>1e-8,sd,torch.ones_like(sd));assert torch.equal(mean,ck[prefix+'mean']) and torch.equal(sd,ck[prefix+'scale'])
original=AttentionHead().to(DEV);original.load_state_dict(ck['head_state']);baseline=evaluate(original,va,ck)
print('Phase A:',json.dumps({k:v for k,v in baseline.items() if k!='curve'}),flush=True)
records=[];states={}
def run(cfg):
 random.seed(42);np.random.seed(42);torch.manual_seed(42)
 if DEV.type=='cuda':torch.cuda.manual_seed_all(42)
 head=TunedHead(cfg['hidden'],cfg['dropout']).to(DEV);opt=torch.optim.AdamW(head.parameters(),lr=cfg['lr'],weight_decay=.0001);tl=loader(tr,ck,True);vl=loader(va,ck);te=loader(tr,ck);best=float('inf');stale=0;hist=[];start=time.perf_counter()
 for e in range(1,36):
  fitloss=fit_epoch(head,tl,ck,cfg,opt);vloss=epoch(head,vl,ck);hist.append({'epoch':e,'training_objective':fitloss,'validation':vloss})
  if vloss['total']<best-.0001:best=vloss['total'];selected=e;state={k:v.detach().cpu().clone() for k,v in head.state_dict().items()};stale=0
  else:stale+=1
  if stale>=7:break
 head.load_state_dict(state);result=evaluate(head,va,ck);record={'config':cfg,'best_epoch':selected,'epochs_run':e,'best_validation_loss':best,'seconds':time.perf_counter()-start,'history':hist,**result};records.append(record);states[cfg['trial']]=state
 print(f"Trial {cfg['trial']} {cfg['variant']} epoch{selected}: F1={result['operating']['f1']:.4f} FPR={result['operating']['false_positive_rate']:.4f} recall={result['operating']['recall']:.4f} MAE={result['magnitude']['mae']:.4f}",flush=True)
 return record
def gate(r):return r['feasible'] and r['magnitude']['mae']<=baseline['magnitude']['mae']+.02 and r['magnitude']['strong_mae']<=baseline['magnitude']['strong_mae']+.10
def rank(r):
 d=r['operating'];m=r['magnitude'];return (d['false_positive_rate'],-d['f1'],m['mae'],m['strong_mae'],m['strong_underprediction_percent'],r['config']['trial'])
def useful(r):
 if not gate(r):return False
 d=r['operating'];z=baseline['operating'];return (d['false_positive_rate']<=z['false_positive_rate']-.02 and d['f1']>=z['f1']-.01) or (d['f1']>=z['f1']+.01 and d['false_positive_rate']<=z['false_positive_rate']+.01) or (r['magnitude']['mae']<=baseline['magnitude']['mae']-.02 and d['false_positive_rate']<=z['false_positive_rate']+.01 and d['f1']>=z['f1']-.01)
for cfg in PLAN:run(cfg)
qualified=[r for r in records if gate(r)];phaseb=min(qualified,key=rank) if qualified else min(records,key=lambda r:r['magnitude']['mae'])
if phaseb['magnitude']['std_ratio']<.8:
 for tid,variant in [(11,'strong_bin'),(12,'asymmetric')]:run({**phaseb['config'],'trial':tid,'variant':variant})
choices=[r for r in records if useful(r)];selected=min(choices,key=rank) if choices else None
bundle={'protocol':PROTOCOL,'plan':PLAN,'baseline_validation':baseline,'trials':records,'selected_trial':selected['config']['trial'] if selected else None,'selected_config':selected['config'] if selected else {'hidden':32,'dropout':.2,'source':'original'},'selected_validation':selected if selected else baseline,'input_hashes':before,'scalers':{k:ck[k] for k in ['cmean','cscale','smean','sscale','target_center','target_scale']},'selected_state':states[selected['config']['trial']] if selected else ck['head_state'],'phase_b_reference_trial':phaseb['config']['trial'],'environment':ck['environment']}
assert all(torch.equal(v.cpu(),cc['state_dict'][k]) for k,v in c.state_dict().items()) and all(torch.equal(v.cpu(),bc['state_dict'][k]) for k,v in b.state_dict().items())
assert all(not p.requires_grad and p.grad is None for model in [c,b] for p in model.parameters())
with (OUT/'models/selected_3s_study.pt').open('xb') as fp:torch.save(bundle,fp)
rows=[]
for r in records:
 row={**r['config'],'selected':r is selected,'best_epoch':r['best_epoch'],'epochs_run':r['epochs_run'],'feasible':r['feasible'],'useful':useful(r),'threshold':r['operating']['threshold']}
 row.update({'val_'+k:r['operating'][k] for k in ['precision','recall','f1','false_positive_rate']});row.update({'val_fixed_'+k:r['fixed'][k] for k in ['precision','recall','f1','false_positive_rate']});row.update({'val_'+k:v for k,v in r['magnitude'].items()});rows.append(row)
with (OUT/'trial_results.csv').open('x',newline='',encoding='utf-8') as fp:
 w=csv.DictWriter(fp,fieldnames=list(rows[0]));w.writeheader();w.writerows(rows)
assert all(sha(Path(p))==v for p,v in before.items())
print('Final validation selection:',bundle['selected_trial'],'(None means original retained); trials',len(records),flush=True)


Phase A: {"fixed": {"accuracy": 0.7756622516556292, "precision": 0.8375893769152196, "recall": 0.6816292601828762, "f1": 0.7516040329972502, "false_positive_rate": 0.13107996702390767, "roc_auc": 0.8647048221710083, "pr_auc_ap": 0.8869448398872987, "confusion_matrix": [[1054, 159], [383, 820]]}, "unconstrained": {"threshold": 0.41, "accuracy": 0.7719370860927153, "precision": 0.7767402376910016, "recall": 0.7605985037406484, "f1": 0.7685846283074339, "false_positive_rate": 0.21681780708985984, "roc_auc": 0.8647048221710083, "pr_auc_ap": 0.8869448398872987, "confusion_matrix": [[950, 263], [288, 915]]}, "operating": {"threshold": 0.41, "accuracy": 0.7719370860927153, "precision": 0.7767402376910016, "recall": 0.7605985037406484, "f1": 0.7685846283074339, "false_positive_rate": 0.21681780708985984, "roc_auc": 0.8647048221710083, "pr_auc_ap": 0.8869448398872987, "confusion_matrix": [[950, 263], [288, 915]]}, "feasible": true, "magnitude": {"n": 1203, "mae": 0.6155476495809389, "rmse": 0.7